<a href="https://colab.research.google.com/github/abdullahab001/Resume-Optimizer/blob/main/Open_AI_Resume_Matcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
import os

In [19]:
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"]) #Replace it with your API Key

def gpt_feedback(resume, job_description):
    prompt = f"""
    You are an expert hiring manager.

    Analyze the resume and job description below. Highlight 3 strengths and 3 weaknesses in terms of fit.

    Resume:
    {resume}

    Job Description:
    {job_description}

    Output format:
    - 3 Strengths
    - 3 Weaknesses
    - One-line Fit Summary
    """

    response = client.chat.completions.create(
    model="gpt-3.5-turbo",  # or "gpt-4"
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": "Match this resume to the job description..."},
    ],
    temperature=0.5,
    max_tokens=500,
)


    return response.choices[0].message.content


In [ ]:
# Install dependencies
!pip install transformers torch gradio --quiet


In [22]:

# Import libraries
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
import gradio as gr

# Load embedding model (offline, from Hugging Face hub)
model_name = "thenlper/gte-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Embedding + similarity functions
def embed(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        return outputs.last_hidden_state.mean(dim=1)

def cosine_similarity(a, b):
    return F.cosine_similarity(a, b).item()

# Combine into main matcher function
def match_resume(resume, job_desc):
    emb1 = embed(resume)
    emb2 = embed(job_desc)
    score = cosine_similarity(emb1, emb2)
    feedback = gpt_feedback(resume, job_desc)

    return f"""🔍 **Match Score:** {round(score * 100, 2)}%

{feedback}
"""

# Gradio UI
gr.Interface(
    fn=match_resume,
    inputs=[
        gr.Textbox(label="Paste Resume", lines=12, placeholder="Your resume text..."),
        gr.Textbox(label="Paste Job Description", lines=12, placeholder="Job description..."),
    ],
    outputs=gr.Markdown(label="Result"),
    title="🧠 Resume Matcher",
    description="Match your resume to a job description using sentence embeddings."
).launch()


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a4930f91a2646ae3ab.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
!pip install transformers --quiet